# Avaliação — detecção de insetos-praga da soja

TCC de Engenharia da Computação, Centro Universitário IBMEC – RJ
Lucas Muniz da Costa Marques · Orientador: Prof. PhD. William de Souza Barbosa

Executa as análises da Seção 2.6 sobre as execuções já treinadas pelo notebook
`treinamento_colab.ipynb`. Não treina nada: só lê os pesos gravados no Drive.

1. **Avaliação por classe e matriz de confusão** de cada uma das 25 execuções,
   com o `best.pt` e com o `last.pt`, para que a escolha entre os dois critérios
   possa ser feita depois sem nova rodada de GPU.
2. **Tempo de inferência na T4**, uma imagem por vez, vinte descartadas para
   aquecimento e cem cronometradas, decomposto em pré-processamento, rede e
   pós-processamento.
3. **Exportação para ONNX** dos três modelos, para a medição em CPU na máquina
   local.

## Antes de executar

**Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4.** O
tempo de inferência só é comparável com a meta de 50 ms se for medido na T4.

Como o de treinamento, este notebook é idempotente: cada avaliação é gravada no
Drive assim que termina, e uma reexecução pula o que já foi feito.

## 1. Ambiente

In [ ]:
import subprocess

GPU = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip()

if "T4" not in GPU:
    print(f"ATENCAO: a GPU conectada e '{GPU}', nao uma T4.")
    print("Os tempos medidos aqui nao serao comparaveis com a meta de 50 ms.")
else:
    print(f"OK: {GPU}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# A mesma versao do treinamento: as metricas, a matriz de confusao e a NMS sao
# calculadas pela biblioteca, e uma versao diferente mudaria os numeros.
VERSAO_ULTRALYTICS = "8.4.183"

subprocess.run(["pip", "install", "-q", f"ultralytics=={VERSAO_ULTRALYTICS}"], check=True)

import ultralytics
ultralytics.checks()

## 2. Configuração e dados

In [ ]:
from pathlib import Path

DRIVE      = Path("/content/drive/MyDrive/TCC")
PACOTE     = DRIVE / "dados_yolo.zip"
DADOS      = Path("/content/dados_yolo")
RUNS_DRIVE = DRIVE / "runs"                  # pesos gravados pelo treinamento
AVAL       = Path("/content/avaliacao")      # saida local
AVAL_DRIVE = DRIVE / "avaliacao"             # saida persistente

DOBRAS  = [1, 2, 3, 4, 5]
MODELOS = ["yolov8n", "yolov8s", "yolov8m"]
MODELO_ESCOLHIDO = "yolov8m"

# As 25 execucoes treinadas: os tres modelos com a semente 0 (fase 1) e o modelo
# escolhido com as sementes 1 e 2 (fase 2).
EXECUCOES = (
    [(m, d, 0) for m in MODELOS for d in DOBRAS]
    + [(MODELO_ESCOLHIDO, d, s) for s in (1, 2) for d in DOBRAS]
)

AVAL_DRIVE.mkdir(parents=True, exist_ok=True)
print(f"pacote ......... {PACOTE}  (existe: {PACOTE.exists()})")
print(f"execucoes ...... {len(EXECUCOES)}")
faltando = [e for e in EXECUCOES
            if not (RUNS_DRIVE / f"{e[0]}_dobra{e[1]}_s{e[2]}" / "concluido.txt").exists()]
assert not faltando, f"execucoes sem concluido.txt: {faltando}"
print("todas as execucoes estao concluidas no Drive")

In [ ]:
import shutil, zipfile, time

if DADOS.exists():
    print(f"{DADOS} ja esta no disco local; nada a fazer.")
else:
    t0 = time.time()
    local_zip = Path("/content/dados_yolo.zip")
    shutil.copy2(PACOTE, local_zip)
    DADOS.mkdir(parents=True)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(DADOS)
    local_zip.unlink()
    print(f"copiado e descompactado em {time.time()-t0:.0f}s")

# Mesmos ajustes do notebook de treinamento: o campo 'path' dos YAML e a forma
# dos caminhos nas listas das dobras. Ambos sao idempotentes.
for dobra in DOBRAS:
    arq = DADOS / f"data_dobra{dobra}.yaml"
    arq.write_text("\n".join(
        f"path: {DADOS}" if linha.startswith("path:") else linha
        for linha in arq.read_text(encoding="utf-8").splitlines()
    ) + "\n", encoding="utf-8")

for arq in sorted((DADOS / "dobras").glob("*.txt")):
    atuais = arq.read_text(encoding="utf-8").split()
    novas = [f"./../images/{linha.rsplit('/', 1)[-1]}" for linha in atuais]
    if novas != atuais:
        arq.write_text("".join(f"{l}\n" for l in novas), encoding="utf-8")
    for linha in novas:
        assert (arq.parent / linha[2:]).exists(), f"{arq.name}: {linha} nao resolve"

CLASSES = [l for l in (DADOS / "classes.txt").read_text(encoding="utf-8").split() if l]
assert len(CLASSES) == 12
print("dados prontos; classes:", ", ".join(CLASSES))

## 3. Avaliação por classe e matriz de confusão

Cada execução é avaliada sobre a própria dobra de validação. As métricas usam o
limiar de confiança da validação do treinamento (0,001); a matriz de confusão usa
0,25, o limiar de uso da predição, porque a 0,001 quase toda saída da rede conta
como detecção. Para cada execução ficam gravados, em um JSON no Drive:

- precisão, revocação, mAP@0,5 e mAP@0,5:0,95 no agregado e por classe;
- o número de imagens e de instâncias de cada classe na dobra, contado direto dos
  rótulos, para que cada valor por classe possa ser julgado pelo tamanho da
  amostra (Seção 2.6.2);
- a matriz de confusão 13 × 13 em contagens brutas: **linhas são a classe
  predita, colunas a classe verdadeira**, e o último índice é o fundo (objeto não
  detectado ou detecção sem objeto correspondente).

A matriz é guardada em contagens, e não normalizada, para que as dobras possam ser
somadas e as classes agrupadas depois — a análise praga × não praga sai dela.

In [ ]:
import json
from collections import Counter

MATRIZ_CONF = 0.25


def amostra_validacao(dobra):
    """Imagens e instancias de cada classe na dobra de validacao."""
    imagens, instancias = Counter(), Counter()
    lista = (DADOS / "dobras" / f"dobra{dobra}_val.txt").read_text(encoding="utf-8").split()
    for linha in lista:
        rotulo = DADOS / "labels" / (Path(linha).stem + ".txt")
        ids = [int(l.split()[0]) for l in rotulo.read_text(encoding="utf-8").splitlines() if l.strip()]
        instancias.update(ids)
        imagens.update(set(ids))
    return imagens, instancias


def avalia(modelo, dobra, semente, peso):
    nome = f"{modelo}_dobra{dobra}_s{semente}"
    destino = AVAL_DRIVE / f"{nome}_{peso}.json"
    if destino.exists():
        print(f"[ja avaliada] {nome} {peso}")
        return

    rede = YOLO(str(RUNS_DRIVE / nome / "weights" / f"{peso}.pt"))
    comum = dict(data=str(DADOS / f"data_dobra{dobra}.yaml"), split="val",
                 imgsz=640, batch=16, verbose=False, project=str(AVAL), exist_ok=True)

    # Duas passagens. As metricas exigem o limiar baixo da validacao (0,001), sem o
    # qual a curva precisao-revocacao fica truncada e o mAP sai subestimado. A
    # matriz de confusao, porem, e montada pela biblioteca com esse mesmo limiar, e
    # a 0,001 ela conta como deteccao quase tudo que a rede emite. Por isso a
    # matriz vem de uma segunda passagem a 0,25, o limiar de uso da predicao.
    m = rede.val(conf=0.001, plots=False, name=f"{nome}_{peso}_metricas", **comum)
    capturado = {}
    rede.add_callback("on_val_end", lambda v: capturado.update(validador=v))
    # plots=True e obrigatorio: sem ele a biblioteca nao preenche a matriz.
    rede.val(conf=MATRIZ_CONF, plots=True, name=f"{nome}_{peso}", **comum)
    matriz = capturado["validador"].confusion_matrix

    imagens, instancias = amostra_validacao(dobra)
    por_classe = {}
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        por_classe[int(c)] = dict(precisao=float(p), revocacao=float(r),
                                  mAP50=float(ap50), mAP50_95=float(ap))

    registro = {
        "execucao": nome, "modelo": modelo, "dobra": dobra, "semente": semente,
        "peso": peso,
        "geral": dict(precisao=float(m.box.mp), revocacao=float(m.box.mr),
                      mAP50=float(m.box.map50), mAP50_95=float(m.box.map)),
        "classes": [
            dict(id=c, classe=CLASSES[c], imagens=imagens[c],
                 instancias=instancias[c], **por_classe.get(c, {}))
            for c in range(len(CLASSES))
        ],
        "matriz": matriz.matrix.astype(int).tolist(),
        "matriz_orientacao": "linhas = predito, colunas = verdadeiro, ultimo indice = fundo",
        "matriz_conf": MATRIZ_CONF,
        "matriz_iou": 0.45,   # limiar de casamento fixo do validador da biblioteca
        "velocidade_ms": {k: float(v) for k, v in m.speed.items()},
        "ultralytics": ultralytics.__version__, "gpu": GPU,
    }
    destino.write_text(json.dumps(registro, ensure_ascii=False, indent=1), encoding="utf-8")

    # As figuras da biblioteca vao junto, para conferencia visual.
    for figura in (AVAL / f"{nome}_{peso}").glob("confusion_matrix*.png"):
        shutil.copy2(figura, AVAL_DRIVE / f"{nome}_{peso}_{figura.name}")

    g = registro["geral"]
    print(f"[avaliada]     {nome} {peso}: mAP50 {g['mAP50']:.3f}  mAP50-95 {g['mAP50_95']:.3f}")

In [ ]:
from ultralytics import YOLO

for modelo, dobra, semente in EXECUCOES:
    for peso in ("best", "last"):
        avalia(modelo, dobra, semente, peso)

## 4. Tempo de inferência na T4

Uma imagem por vez, por ser a condição de uso numa plataforma de monitoramento.
As imagens são lidas do disco antes da medição, para que o tempo de leitura do
arquivo não entre na conta. Das 120 primeiras imagens de validação da dobra 1, as
vinte primeiras aquecem a GPU e são descartadas; as cem seguintes são
cronometradas.

A biblioteca separa o tempo em pré-processamento, rede e pós-processamento — este
último inclui a supressão não máxima, que depende dos limiares (aqui os padrões da
predição, `conf=0.25` e `iou=0.7`). O tempo total de parede, medido em volta da
chamada inteira, é registrado à parte.

Mede-se o `best.pt` da dobra 1, semente 0, de cada modelo: o tempo depende da
arquitetura, não dos pesos.

In [ ]:
import cv2
import pandas as pd

AQUECIMENTO, MEDIDAS = 20, 100

lista = (DADOS / "dobras" / "dobra1_val.txt").read_text(encoding="utf-8").split()
quadros = [cv2.imread(str(DADOS / "dobras" / l[2:])) for l in lista[:AQUECIMENTO + MEDIDAS]]
assert all(q is not None for q in quadros)

arq_tempos = AVAL_DRIVE / "tempo_t4.csv"
if arq_tempos.exists():
    tempos = pd.read_csv(arq_tempos)
    print("tempos ja medidos; lidos do Drive.")
else:
    linhas = []
    for modelo in MODELOS:
        rede = YOLO(str(RUNS_DRIVE / f"{modelo}_dobra1_s0" / "weights" / "best.pt"))
        for i, quadro in enumerate(quadros):
            t0 = time.perf_counter()
            r = rede.predict(quadro, imgsz=640, device=0, verbose=False)[0]
            total = (time.perf_counter() - t0) * 1000
            if i >= AQUECIMENTO:
                linhas.append(dict(modelo=modelo, imagem=i, **r.speed, total=total))
    tempos = pd.DataFrame(linhas)
    tempos.to_csv(arq_tempos, index=False)

# Mediana e intervalo interquartil, como pede a Secao 2.6.3.
resumo = tempos.groupby("modelo")[["preprocess", "inference", "postprocess", "total"]].quantile(
    [0.25, 0.5, 0.75]).unstack().round(2)
print(resumo)
resumo.to_csv(AVAL_DRIVE / "tempo_t4_resumo.csv")

## 5. Exportação para ONNX

Os três modelos são exportados para ONNX com tamanho de entrada fixo de 640 ×
640. Os arquivos ficam em `MyDrive/TCC/avaliacao/` e devem ser baixados para a
máquina local, onde é medido o tempo em CPU.

In [ ]:
for modelo in MODELOS:
    destino = AVAL_DRIVE / f"{modelo}_dobra1_s0_best.onnx"
    if destino.exists():
        print(f"[ja exportado] {destino.name}")
        continue
    # A copia local evita que a exportacao grave arquivos dentro da pasta de
    # pesos do treinamento no Drive.
    local = AVAL / f"{modelo}_dobra1_s0_best.pt"
    local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RUNS_DRIVE / f"{modelo}_dobra1_s0" / "weights" / "best.pt", local)
    caminho = YOLO(str(local)).export(format="onnx", imgsz=640, dynamic=False, simplify=True)
    shutil.copy2(caminho, destino)
    print(f"[exportado]    {destino.name}  ({destino.stat().st_size / 1e6:.1f} MB)")